# Stage 6: Backend REST API Development & Testing (Beginner's Guide)
### E-Commerce Intelligence & Decision Support Platform

---

### What is a "Backend REST API"?
So far, we have cleaned data, trained Machine Learning models, and evaluated them.
**The Backend API is the bridge** that connects our ML models to the outside world (websites, mobile apps, or management dashboards):
- When a website visitor logs in with their `CustomerID`, the website sends a request to our API:
  > *"Hey API, give me the Top-5 recommendations for Customer 16128 using the Decision Tree model!"*
- Our API looks up the model predictions and instantly returns the 5 products in clean **JSON** format!

---

### In this notebook, we will:
1. Explore the architecture of our `backend/` folder.
2. Test all 7 endpoints using FastAPI's `TestClient` (no terminal server needed to test!).
3. Inspect sample outputs for **Recommendations**, **Customer RFM Profiles**, and **Platform KPIs**.
4. Learn how to launch the live server and view the interactive **Swagger UI Documentation** at `http://localhost:8000/docs`.


### Step 1: Import FastAPI TestClient & Load Backend
FastAPI includes a built-in `TestClient` that lets us simulate web requests directly inside Python!

In [1]:
import sys
import os
import json
import pandas as pd

# Add project root to path
if not os.path.exists("backend"):
    sys.path.insert(0, "..")
else:
    sys.path.insert(0, ".")

from fastapi.testclient import TestClient
from backend.main import app
from backend.database.data_loader import db

# Load data into memory
db.load_all()
client = TestClient(app)

print("Backend API loaded and ready for testing!")


[Backend] Loading data and machine learning models into memory...
  [OK] Loaded 4,933 customer profiles.
  [OK] Loaded 4,235 catalog products.


[Backend] All datasets and models ready in memory!

Backend API loaded and ready for testing!


### Step 2: Test Endpoint 1 – Health Check (`GET /api/health`)
Verifies that the backend server is running and healthy.

In [2]:
response = client.get("/api/health")
print(f"Status Code: {response.status_code}")
print("Response JSON:")
print(json.dumps(response.json(), indent=2))


Status Code: 200
Response JSON:
{
  "status": "healthy",
  "service": "E-Commerce Intelligence & Decision Support API",
  "version": "1.0.0"
}


### Step 3: Test Endpoint 2 – Business Overview KPIs (`GET /api/kpis`)
Returns high-level business intelligence metrics: total customers, products, revenue, and active ML models.

In [3]:
response = client.get("/api/kpis")
kpis = response.json()

print("Platform High-Level KPIs:")
print(f"  Total Customers:       {kpis['total_customers']:,}")
print(f"  Total Catalog SKUs:    {kpis['total_products']:,}")
print(f"  Total Orders:          {kpis['total_orders']:,}")
print(f"  Total Revenue:         £{kpis['total_revenue_gbp']:,.2f}")
print(f"  Average Order Value:   £{kpis['average_order_value_gbp']:,.2f}")
print(f"  Active Models:         {', '.join(kpis['ml_models_active'])}")


Platform High-Level KPIs:
  Total Customers:       4,933
  Total Catalog SKUs:    4,235
  Total Orders:          25,846
  Total Revenue:         £11,811,997.56
  Average Order Value:   £371.36
  Active Models:         Decision Tree, Logistic Regression, Popularity Baseline


### Step 4: Test Endpoint 3 – Personalized Recommendations (`GET /api/recommendations/{id}`)
Let's ask for the Top-5 personalized recommendations for Customer `16128` using our best model (**Decision Tree**):

In [4]:
response = client.get("/api/recommendations/16128?model=decision_tree")
data = response.json()

print(f"Customer ID:      {data['customer_id']}")
print(f"Model Applied:    {data['model_used']}")
print(f"Cold-Start Used?  {data['is_cold_start_fallback']}")
print()
print("Top 5 Personalized Product Recommendations:")

df_recs = pd.DataFrame(data["recommendations"])
df_recs[["rank", "stock_code", "description", "avg_unit_price", "purchase_probability", "bought_in_evaluation"]]


Customer ID:      16128
Model Applied:    Decision Tree (Personalized ML)
Cold-Start Used?  False

Top 5 Personalized Product Recommendations:


,rank,stock_code,description,avg_unit_price,purchase_probability,bought_in_evaluation
0,1,85152,HAND OVER THE CHOCOLATE SIGN,2.07,0.9435,False
1,2,22111,SCOTTIE DOG HOT WATER BOTTLE,4.89,0.9435,False
2,3,21754,HOME BUILDING BLOCK WORD,5.93,0.9435,False
3,4,22507,MEMO BOARD RETROSPOT DESIGN,4.90,0.9435,False
4,5,22197,SMALL POPCORN HOLDER,0.84,0.9435,False


### Step 5: Test Endpoint 4 – Customer RFM Intelligence Profile (`GET /api/customers/{id}`)
Let's retrieve the detailed spending, order frequency, and segmentation profile for Customer `12347`:

In [5]:
response = client.get("/api/customers/12347")
cust = response.json()

print("Customer Profile & Segmentation:")
print(f"  Customer ID:       {cust['customer_id']}")
print(f"  Assigned Segment:  {cust['segment']}")
print(f"  Country:           {cust['country']}")
print(f"  Total Spending:    £{cust['total_spending_gbp']:,.2f}")
print(f"  Total Orders:      {cust['total_orders']}")
print(f"  Avg Order Spend:   £{cust['avg_order_value_gbp']:,.2f}")
print(f"  Recency:           {cust['recency_days']} days ago")
print(f"  Tenure:            {cust['tenure_days']} days with us")


Customer Profile & Segmentation:
  Customer ID:       12347
  Assigned Segment:  VIP Champion (High Spend, High Activity)
  Country:           Iceland
  Total Spending:    £2,434.96
  Total Orders:      4
  Avg Order Spend:   £608.74
  Recency:           54 days ago
  Tenure:            212 days with us


### Step 6: Test Endpoint 5 – Model Benchmarks (`GET /api/models/benchmarks`)
Provides real-time model evaluation metrics and confidence intervals directly to decision-makers:

In [6]:
response = client.get("/api/models/benchmarks")
benchmarks = response.json()

df_benchmarks = pd.DataFrame(benchmarks["models"])
df_benchmarks[["model", "hits", "total_customers", "hit_rate_at_5_pct", "ci_95_lower_pct", "ci_95_upper_pct", "improvement_vs_level1_pct"]]


,model,hits,total_customers,hit_rate_at_5_pct,ci_95_lower_pct,ci_95_upper_pct,improvement_vs_level1_pct
0,Level 0 - Random,146,781,18.69,16.12,21.58,-60.86
1,Level 1 - Popularity,373,781,47.76,44.28,51.26,0.00
2,Logistic Regression,586,781,75.03,71.88,77.94,57.10
3,Decision Tree,594,781,76.06,72.94,78.92,59.25


### Step 7: How to Launch the Live Web Server
To start the live backend web server on your machine:

1. Open your terminal in the project folder.
2. Run this command:
   ```bash
   uvicorn backend.main:app --reload --port 8000
   ```
3. Open your browser and go to:
   - **Interactive Swagger Documentation:** [http://localhost:8000/docs](http://localhost:8000/docs)
   - You can test every endpoint interactively with one click using the **"Try it out"** button!
